# 03 — From evidence to a cited answer

**10–15 minute lab.** Build one typed evidence result, generate a cited answer through the real
`GenerationPipeline`, then prove that an invented citation fails closed.

**Flow:** evidence → cited answer → fail closed.


In [ ]:
from raglab import Citation, ProvenanceStatus
from raglab.errors import GenerationError
from raglab.generation import (
    GenerationConfig,
    GenerationPipeline,
    GenerationRequest,
    ModelInvocation,
)
from raglab.retrieval import (
    CollectionMetadata,
    RankingTrace,
    RetrievalRequest,
    RetrievalResponse,
    RetrievalResult,
)


class StaticRetrieval:
    def __init__(self, results):
        self.results = results

    def collection_metadata(self, collection):
        return CollectionMetadata(collection, "hermetic-embedding", 2)

    def retrieve(self, request):
        return RetrievalResponse(
            query=request.query,
            rewritten_query=None,
            query_variants=(request.query,),
            filters=request.filters,
            results=self.results,
        )

class FixedModel:
    def __init__(self, answer):
        self.answer = answer

    def generate(self, prompt, *, system, schema, config):
        return ModelInvocation({"answer": self.answer, "abstained": False}, 80, 18)


## Checkpoint 1 — Objective: create grounded evidence

**Run:** construct the same public retrieval contract used by the production pipeline.


In [ ]:
citation = Citation(
    source_uri="memory://aster-manual",
    source_name="aster-manual.md",
    title="Aster Greenhouse Controller",
    heading_path=("Fault E17",),
    start_page=None,
    end_page=None,
    start_line=42,
    end_line=45,
    provenance_status=ProvenanceStatus.COMPLETE,
)
evidence = RetrievalResult(
    id="e17-evidence",
    document_id="aster-manual",
    content="Fault E17 means irrigation flow stayed below the safe threshold.",
    citation=citation,
    matched_chunk_ids=("chunk-7",),
    first_chunk_index=7,
    last_chunk_index=7,
    trace=RankingTrace(1, 1, 0.05, 9.2, 0.032, None, None),
)
print(evidence.content)
print(
    f"Source: {evidence.citation.source_name}, "
    f"lines {evidence.citation.start_line}-{evidence.citation.end_line}"
)


### What to observe

Expect full evidence text plus traceable source lines. The model does not receive an anonymous
snippet; it receives evidence that can support a citation.

### Conclusion

Grounding starts before generation: retrieval must preserve both content and provenance.


## Checkpoint 2 — Objective: generate and validate a cited answer

**Run:** use a deterministic model adapter while keeping RAGLab's real orchestration and
citation validator.


In [ ]:
request = GenerationRequest(
    retrieval=RetrievalRequest("What does fault E17 mean?", collection="lab"),
    config=GenerationConfig(minimum_sources=1),
)
pipeline = GenerationPipeline(
    StaticRetrieval((evidence,)),
    FixedModel("E17 reports irrigation flow below the safe threshold [S1]."),
    embedding_model="hermetic-embedding",
    embedding_dimension=2,
)
response = pipeline.generate(request)
print(response.answer)
print("Validated sources:", [source.id for source in response.sources])


### What to observe

Expect `[S1]` in the answer and one validated source. The adapter controls only model output;
the production pipeline still assigns and checks source IDs.

### Conclusion

A cited answer is accepted only when every source ID maps back to retrieved evidence.


## Checkpoint 3 — Objective: fail closed on citation drift

**Run:** make the model invent `[S99]` and observe the domain error.


In [ ]:
unsafe_pipeline = GenerationPipeline(
    StaticRetrieval((evidence,)),
    FixedModel("The controller should be replaced immediately [S99]."),
    embedding_model="hermetic-embedding",
    embedding_dimension=2,
)
try:
    unsafe_pipeline.generate(request)
except GenerationError as error:
    print(f"Blocked: {error}")
else:
    raise AssertionError("An unknown citation must never escape validation")


### What to observe

Expect a `Blocked:` message and no answer response. Prompt instructions alone are not the safety
boundary; independent validation is.

### Conclusion

Strict RAG fails closed: unsupported citation IDs stop the response instead of becoming output.

## Optional appendix — live local generation

Use `raglab-generate` with an indexed collection to explore Ollama, source shortfall, and
hierarchical synthesis. Those slower service boundaries do not block this core lesson.
